# Haruka v2: the default model and how every figure was fitted

Every figure in the paper uses one model configuration: the **composition-gated shared-decoder Haruka**
(`bench_kras_xenium/run_haruka_separate_batch_heads.py`, which wraps `bench_mouse_brain_scviva/run_haruka_shared_decoder.py`).

* **z (background)**: per-cell latent shared by control and condition cells.
* **s (salient)**: condition-specific latent. The decoder sets s = 0 for control cells, so control s is never used
  and every z-vs-s readout is computed on condition-arm cells only.
* **Decoders**: a negative-binomial expression decoder plus two niche heads, the cell-type composition of the k = 20
  spatial neighbours and the composition-gated neighbour mean expression (`spatial_weight 100`).
* **Salient composition shift**: D([z, s]) − D([z, 0]), what s adds to the predicted neighbourhood.
* **z/s independence**: one-way ridge penalty (z ← s, weight 10).
* **Batch**: injected into the decoders only (`decoder_batch_dim 8`); separate batch projectors for the
  composition and neighbour-mean heads. z and s encoders are batch-blind.
* **Training**: 150 epochs, 32-d latents, sparse augmentation.

In [ ]:
import sys, pathlib; sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / 'notebooks'] if (p / 'repro.py').exists())))
from repro import run, require, show, fetch, fetch_fitted, STAGES, H, WK, PY, PY_VIO, GPU
# Default: download inputs + the paper's fitted Haruka outputs from Hugging Face, then analyses -> figure.
# The Haruka fit step reuses the unpacked fit; HARUKA_REFIT=1 (env) refits from scratch. Auxiliary fits (baselines, seeds) still run.
# Redraw only from saved outputs: STAGES.update(download=False, fit=False, analysis=False)  (or env HARUKA_FIGURE_ONLY=1)
# Rebuild the inputs from the raw public releases instead of Hugging Face: STAGES.update(data=True)
print(H, STAGES)

## Setup (once per machine)

```bash
git clone -b v2 https://github.com/C0nc/haruka_v2 && cd haruka_v2
conda create -n haruka2 python=3.11 -y && conda activate haruka2
pip install -r requirements.txt
bash install_contrastivevi_patch.sh            # the Haruka model: patched scvi ContrastiveVI module
python -m ipykernel install --user --name haruka2
```
Then open any notebook: section 0 downloads its inputs from Hugging Face and the notebook runs end to end.
The check below confirms the installed scvi carries the patch.

In [ ]:
import os, filecmp, scvi
dst = os.path.join(os.path.dirname(scvi.__file__), 'external', 'contrastivevi')
bad = [f for f in os.listdir(H / 'contrastivevi_patch') if f.endswith('.py') and not filecmp.cmp(H / 'contrastivevi_patch' / f, os.path.join(dst, f), shallow=False)]
print('scvi', scvi.__version__, '| patch installed' if not bad else f'| NOT patched: {bad} -> run install_contrastivevi_patch.sh')

## Default arguments (shared by all fits)

In [ ]:
DEFAULT = """--seed 0 --max_epochs 150 \
  --n_latent 32 --n_hidden 256 --n_layers 1 --decoder_layers 2 --k_niche 20 --neighbor_embedding_dim 16 \
  --composition_loss_weight 1 --neighbor_mean_loss_weight 1 --neighbor_expression_mode composition_gated --spatial_weight 100 \
  --background_wasserstein_penalty 0 --zs_independence_penalty 10 --decoder_batch_dim 8 --batch_injection_mode decoder_only \
  --augmentation_mode sparse --num_workers 4 --direct_only"""
print(DEFAULT)

## Per-dataset arguments

| figure | data | batch key | condition (target) | cell type key | likelihood / HVG | extra |
|---|---|---|---|---|---|---|
| Fig. 2 benchmark | paired mouse-brain simulation (5 section pairs) | slice | disease | cell type | NB | λ sweep, λ = 10 shown |
| Fig. 3 AD plaque | STARmap AD, CODEX CNS | slice / donor | disease | cell type | Normal, pretransformed | |
| Exercise | old control vs old exercise MERFISH | run | exercise | cell type | NB | |
| PEtracer | M2-1 lineage section | graph | malignant cells in tumours | cell type | NB | clone never seen |
| Lung fibrosis | GSE250346 Xenium, 1.63 M cells, 343 genes | core (`sample`) | fibrosis | cell_type | NB, all genes | |
| KRAS inhibitor | Xenium 5K, 920 k cells | slice | KRASi_treated | celltype | NB, 2000 HVG | 3 shuffled-s nulls |

The exact command for each is in that figure's notebook.

## Batch runner used for the real-data fits

In [ ]:
print(open(H / 'bench_real_discovery/run_all_compgated_samplebatch_sepheads.sh').read())

## Outputs of a fit (`<out_root>/<data stem>/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/`)

`background.npy` (z), `salient.npy` (s), `obs_names.npy`, `background_composition.npy` D([z,0]),
`condition_composition.npy` D([z,s]), `salient_composition_shift.npy`, `meta.json` (all arguments + timings),
optionally `model/` and `null_shuffled_s_composition_shift_{k}.npy`.

In [ ]:
import json
m = json.load(open(H / 'notebooks/kras_default_fit_meta.json'))   # copy of the KRAS fit's meta.json
{k: m[k] for k in list(m)[:40] if not isinstance(m[k], (list, dict))}

## Figure